# Full Monte Carlo for Structure Formation from Lattice Defects

Simulates defect formation and computes the power spectrum P(k).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters (scale up N_defects for production runs)
N_defects = 100000  # start small; increase to 1e7+ on cloud
box_size = 1000.0   # Mpc (simulation volume)
defect_size = 5.0   # Mpc (typical defect correlation length)

# Generate defects with lattice-like clustering
np.random.seed(42)
positions = np.random.uniform(0, box_size, (N_defects, 3))

# Add correlation (simple clustering)
for i in range(1000):
    idx = np.random.randint(0, N_defects)
    positions[idx] += np.random.normal(0, defect_size, 3)
    positions[idx] = np.clip(positions[idx], 0, box_size)

# Compute density field on grid (simple binning)
grid_size = 64  # increase for production (128–256)
density = np.zeros((grid_size, grid_size, grid_size))
for pos in positions:
    idx = (pos / box_size * grid_size).astype(int)
    density[tuple(idx)] += 1

# Compute power spectrum (FFT)
density_ft = np.fft.fftn(density)
power = np.abs(density_ft)**2
k = np.fft.fftfreq(grid_size, d=box_size/grid_size) * 2 * np.pi

print(f"Simulation completed with {N_defects} defects")
print(f"Grid size: {grid_size}^3")
print(f"Mean density: {density.mean():.4f}")

# Plot 1D power spectrum (radial average)
plt.figure(figsize=(8,5))
plt.loglog(k[1:grid_size//2], power[1:grid_size//2].mean(axis=(1,2)), label='P(k) from defects')
plt.xlabel('k (Mpc^{-1})')
plt.ylabel('P(k)')
plt.title('Power Spectrum from Lattice Defects')
plt.grid(True)
plt.legend()
plt.savefig('../figures/power-spectrum.pdf')
plt.show()

## Notes
- This is a simplified 3D defect simulation.
- Increase N_defects and grid_size for production runs on cloud.
- Next: add realistic Capotauro defect formation rules.